In [2]:
import os 
import geopandas as gpd
import pandas as pd
from shapely import wkt
import geopy.distance
from shapely.geometry import Point
from tqdm import tqdm
from datetime import date
import matplotlib.pyplot as plt

from mapping_functions import *

In [3]:
# Define the scenario demand settings
# File name of the scenario input file and output file of aggregated
Date = str(date.today())
# Path and file from Demand and Supply at each sector
input_file_path = os.path.join('..', '..','01_data', '01_input_data', '01_raw')
input_file = '\\00_Scenario_BaseCase_Final.xlsx'  
full_input_path = os.path.abspath(os.path.join(os.getcwd(), input_file_path + input_file))

topo_file_path = os.path.join('..', '..','01_data', '01_input_data', '02_processed')
topo_file = '\\input_network_data.xlsx'
full_topo_path = os.path.abspath(os.path.join(os.getcwd(), topo_file_path + topo_file))

output_file_path = os.path.join('..', '..','01_data', '01_input_data', '02_processed')
output_file = '\\Demand_Nodes_'+Date+'.csv'  
full_output_path = os.path.abspath(os.path.join(os.getcwd(), output_file_path + output_file))


In [4]:
#Setting for Script 
Save_Output = False
Visualisation = True

In [5]:
#Binary Scenario Settings
#Industry_overall
Industry = False 

#Demand sectors
Refineries = False
Chemicals = False
Steel = False 
Paper = False
Mineral_Processing = False 
Metal_Processing = False
Non_Metallic_Minerals = False

Other_Industry = True                # NUTS3 region and not coordinates 

Residential_Heat = False             # NUTS3 region and not coordinates 
District_Heat = False                # PLZ code and not coordinates 

Passenger = False                    # NUTS3 region and not coordinates
Public = False                       # NUTS3 region and not coordinates

Air = False
Train = False                        # PLZ code and not coordinates
Trucks = False

In [6]:
topo_file = pd.read_excel(full_topo_path)
source_df = topo_file[['source_name', 'source']].drop_duplicates().rename(columns={'source_name':'node_id', 'source':'geometry'})
target_df = topo_file[['target_name', 'target']].drop_duplicates().rename(columns={'target_name':'node_id', 'target':'geometry'})
nodes = pd.concat([source_df, target_df], ignore_index=True)

nodes['geometry'] = (nodes['geometry'].astype(str).apply(tuple_to_wkt_point).apply(wkt.loads))

nodes_gdf = gpd.GeoDataFrame(nodes, crs='EPSG:3857')
nodes_gdf.to_crs('EPSG:4326', inplace = True)
nodes_gdf['Longitude'] = nodes_gdf.geometry.apply(lambda p: p.x)
nodes_gdf['Latitude'] = nodes_gdf.geometry.apply(lambda p: p.y)
nodes_gdf['Supply'] = 0
nodes_gdf['Demand'] = 0 

#### PLZ Shapefile
Downloaded from Opendatasoft:  
https://public.opendatasoft.com/explore/dataset/georef-germany-postleitzahl/export/

- Load the file (GeoJSON, csv or Excel)
- Check geometry type and postal codes
- Identify any missing plz
- Fill in the missing geometries if possible

In [7]:
# Load the PLZ shapefile and geometry
geojson_path = '../../01_data/01_input_data/01_raw/plz_shape_file/georef-germany-postleitzahl.geojson'
excel_path = '../../01_data/01_input_data/01_raw/plz_shape_file/georef-germany-postleitzahl.xlsx'
plz_df = pd.read_excel(excel_path)

# Clean the PLZ codes 
plz_df['Postleitzahl / Post code'] = clean_plz(plz_df['Postleitzahl / Post code'])

# Convert the DataFrame to a GeoDataFrame
plz_gdf = gpd.GeoDataFrame(plz_df, geometry=gpd.points_from_xy(plz_df['geo_point_2d'].str.extract(r'(\d+\.\d+), (\d+\.\d+)')[1].astype(float), plz_df['geo_point_2d'].str.extract(r'(\d+\.\d+), (\d+\.\d+)')[0].astype(float)), crs='EPSG:4326')

In [8]:
plz_gdf.head(4)

,Name,PLZ Name (short),PLZ Name (long),Geometry,Postleitzahl / Post code,Kreis code,Land name,Land code,Kreis name,geo_point_2d,geometry
0,47551,Bedburg-Hau,47551 Bedburg-Hau,"{""coordinates"":[[[6.1156615,51.7419192],[6.121...",47551,5154,Nordrhein-Westfalen,5,Kreis Kleve,"51.7581345416, 6.20695861505",POINT (6.20696 51.75813)
1,52477,Alsdorf,52477 Alsdorf,"{""coordinates"":[[[6.1218109,50.8594794],[6.125...",52477,5334,Nordrhein-Westfalen,5,Kreis Städteregion Aachen,"50.86861665, 6.17550818828",POINT (6.17551 50.86862)
2,52159,Roetgen,52159 Roetgen,"{""coordinates"":[[[6.1661074,50.6618499],[6.167...",52159,5334,Nordrhein-Westfalen,5,Kreis Städteregion Aachen,"50.6583845345, 6.18337503317",POINT (6.18338 50.65838)
3,52156,Monschau,52156 Monschau,"{""coordinates"":[[[6.1740833,50.5563913],[6.174...",52156,5334,Nordrhein-Westfalen,5,Kreis Städteregion Aachen,"50.5630704175, 6.20863299366",POINT (6.20863 50.56307)


#### Sources for Missing PLZ Codes
For the missing PLZ codes, the following sources were used to find the necessary coordinates:

- **12861 (Berlin-Marzahn)** from Wikipedia
- **51368 (Leverkusen)**: data from [PLZ-Guru](https://www.plz-guru.de/plz/51368)
- **63784 (Obernburg am Main)**: data from [Latitude and Longitude Finder](https://latitudelongitude.org/de/obernburg-am-main/#google_vignette)
- **67056 (Ludwigshafen am Rhein)**: similarly from [Latitude and Longitude Finder](https://latitudelongitude.org/de/ludwigshafen-am-rhein/?utm_source=chatgpt.com)
- **71059 (Sindelfingen)**: from Wikipedia
- **01956 (Senftenberg)**: from [Free Country Maps](https://www.freecountrymaps.com/map/towns/germany/1676297307/)
- **28023 (Bremen)**: from github German plz code sources
- **63659 (Stockheim, Glauburg)**: from GitHub sources
- **67102 (Zeitz)**: from Wikipedia
- **14627 (Elstal Wustermark)**: from GitHub sources

In [9]:
# Data for missing PLZ codes
missing_plz_data = [
    {'Postleitzahl / Post code': '12861', 'city': 'Berlin-Marzahn', 'geo_point_2d': '52.5500, 13.5500'},
    {'Postleitzahl / Post code': '51368', 'city': 'Leverkusen', 'geo_point_2d': '51.0377, 6.9865'},
    {'Postleitzahl / Post code': '63784', 'city': 'Obernburg am Main', 'geo_point_2d': '49.83577, 9.13101'},
    {'Postleitzahl / Post code': '67056', 'city': 'Ludwigshafen am Rhein', 'geo_point_2d': '49.48121, 8.44641'},
    {'Postleitzahl / Post code': '71059', 'city': 'Sindelfingen', 'geo_point_2d': '48.70746, 9.00441'},
    {'Postleitzahl / Post code': '01956', 'city': 'Senftenberg', 'geo_point_2d': '51.5192, 14.0047'},
    {'Postleitzahl / Post code': '28023', 'city': 'Bremen', 'geo_point_2d': '53.0736, 8.8064'},
    {'Postleitzahl / Post code': '63659', 'city': 'Stockheim (Glauburg)', 'geo_point_2d': '50.3246, 9.0176'},
    {'Postleitzahl / Post code': '67102', 'city': 'Zeitz', 'geo_point_2d': '51.0478, 12.1383'},
    {'Postleitzahl / Post code': '14627', 'city': 'Elstal (Wustermark)','geo_point_2d': '52.5400, 12.9900'}
]

In [10]:
# Create a GeoDataFrame like above 
missing_plz_df = pd.DataFrame(missing_plz_data)
missing_plz_df['Postleitzahl / Post code'] = clean_plz(missing_plz_df['Postleitzahl / Post code'])
missing_plz_df['geometry'] = missing_plz_df['geo_point_2d'].apply(lambda x: Point(float(x.split(', ')[1]), float(x.split(', ')[0])))
missing_plz_gdf = gpd.GeoDataFrame(missing_plz_df, geometry='geometry', crs='EPSG:4326')

In [11]:
# Merge with the existing plz shape file
plz_gdf = gpd.GeoDataFrame(pd.concat([plz_gdf, missing_plz_gdf], ignore_index=True), crs='EPSG:4326')

#### Case of NUTS3 regions 
For the data with NUTS3 region, Voronoi polygons are used to aggregate level on a smaller scale 

- Load a NUTS3 shape file for Germany 
- Create Voronoi polygons around these regions
- 

In [12]:
# Load the shapefile for nuts3 region
nuts3_folder = os.path.abspath('../../01_data/01_input_data/01_raw/germany_nuts3')
nuts3_file = 'NUTS250_N3.shp'
nuts3_path = os.path.join(nuts3_folder, nuts3_file)

nuts3_gdf = gpd.read_file(nuts3_path)
nuts3_gdf = nuts3_gdf.to_crs('EPSG:4326')
nuts3_gdf.head(10)

,OBJID,BEGINN,GF,NUTS_LEVEL,NUTS_CODE,NUTS_NAME,geometry
0,DEBKGNU200000001,2023-10-04,4,3,DE111,"Stuttgart, Stadtkreis","POLYGON ((9.22518 48.86601, 9.225 48.86485, 9...."
1,DEBKGNU200000002,2022-12-20,4,3,DE112,Böblingen,"POLYGON ((9.10336 48.69793, 9.10429 48.69674, ..."
2,DEBKGNU200000003,2022-12-20,4,3,DE113,Esslingen,"MULTIPOLYGON (((9.16875 48.60398, 9.16961 48.6..."
3,DEBKGNU200000004,2023-10-04,4,3,DE114,Göppingen,"POLYGON ((9.63255 48.53698, 9.62717 48.53731, ..."
4,DEBKGNU200000005,2023-10-04,4,3,DE115,Ludwigsburg,"MULTIPOLYGON (((9.41396 49.06157, 9.41504 49.0..."
5,DEBKGNU200000006,2023-10-04,4,3,DE116,Rems-Murr-Kreis,"POLYGON ((9.49274 48.76732, 9.49177 48.76676, ..."
6,DEBKGNU200000007,2022-12-20,4,3,DE117,"Heilbronn, Stadtkreis","POLYGON ((9.11743 49.20957, 9.11915 49.20854, ..."
7,DEBKGNU200000008,2023-10-04,4,3,DE118,"Heilbronn, Landkreis","POLYGON ((9.34351 49.17051, 9.34826 49.17122, ..."
8,DEBKGNU200000009,2022-12-20,4,3,DE119,Hohenlohekreis,"POLYGON ((9.83442 49.28466, 9.83478 49.28344, ..."
9,DEBKGNU20000000A,2023-10-04,4,3,DE11A,Schwäbisch Hall,"POLYGON ((10.16737 49.05873, 10.16673 49.05849..."


In [13]:
# Check for all NUTS codes beginning with 'DEG' 
deg_codes = nuts3_gdf[nuts3_gdf['NUTS_CODE'].str.startswith('DEG')]
unique_deg_codes = deg_codes['NUTS_CODE'].unique()
print("Unique DEG NUTS codes:", unique_deg_codes)

Unique DEG NUTS codes: ['DEG01' 'DEG02' 'DEG03' 'DEG05' 'DEG06' 'DEG07' 'DEG09' 'DEG0A' 'DEG0C'
 'DEG0D' 'DEG0E' 'DEG0G' 'DEG0J' 'DEG0K' 'DEG0L' 'DEG0M' 'DEG0Q' 'DEG0R'
 'DEG0S' 'DEG0T' 'DEG0U' 'DEG0V']


In [16]:
# Define the path to the directory containing the shapefile components
shapefile_dir = os.path.abspath('../../01_data/01_input_data/01_raw/NUTS_eurostat')

# Load the shapefile using the base name without extension
shapefile_base = 'NUTS_RG_60M_2021_4326.shp'
shapefile_path = os.path.join(shapefile_dir, shapefile_base)

try:
    nuts3_gdf = gpd.read_file(shapefile_path)
    print("Shapefile loaded successfully.")

    # Display the first few rows to understand the structure
    print(nuts3_gdf.head())

    # Check column names to identify the NUTS code column
    print("Columns in nuts3_gdf:", nuts3_gdf.columns)

    # Assuming the NUTS code column is named 'NUTS_ID' or similar
    if 'NUTS_ID' in nuts3_gdf.columns:
        # Filter for German NUTS regions (those starting with 'DE')
        german_nuts3_gdf = nuts3_gdf[nuts3_gdf['NUTS_ID'].str.startswith('DE')]

        # Display the filtered GeoDataFrame
        print("German NUTS3 regions:")
        print(german_nuts3_gdf.head())
    else:
        print("The 'NUTS_ID' column does not exist in the nuts3_gdf.")

except Exception as e:
    print("An error occurred:", e)


Shapefile loaded successfully.
  NUTS_ID  LEVL_CODE CNTR_CODE                     NAME_LATN  \
0   DE149          3        DE                   Sigmaringen   
1   DE211          3        DE  Ingolstadt, Kreisfreie Stadt   
2   DE212          3        DE     München, Kreisfreie Stadt   
3   DE213          3        DE   Rosenheim, Kreisfreie Stadt   
4   DE214          3        DE                     Altötting   

                      NUTS_NAME  MOUNT_TYPE  URBN_TYPE  COAST_TYPE    FID  \
0                   Sigmaringen         4.0          3           3  DE149   
1  Ingolstadt, Kreisfreie Stadt         4.0          2           3  DE211   
2     München, Kreisfreie Stadt         4.0          1           3  DE212   
3   Rosenheim, Kreisfreie Stadt         4.0          2           3  DE213   
4                     Altötting         4.0          2           3  DE214   

                                            geometry  
0  POLYGON ((9.3476 48.2395, 9.6049 48.0023, 9.39...  
1  POLYGON 

#### Aggregate demand and create network

- Call necessary sectors sheet
- Run through each sector and find closest nodes to aggregate peak load demand 
- Check the validity of demand aggregatin 
- Visualize on the map 

In [17]:
# AggregateDemand function requires 'ID' column name
nodes_gdf.rename(columns={'node_id': 'ID'}, inplace=True)

# Use defined sectors to test
sheet_flags = {
    'Ind_Input': Industry,
    'Refineries_Input': Refineries,
    'Chemicals_Input': Chemicals,
    'Steel_Input': Steel,
    'Paper_Input': Paper,
    'Mineral_Processing_Input': Mineral_Processing,
    'Metal_Processing_Input': Metal_Processing,
    'Non_Metallic_Minerals_Input': Non_Metallic_Minerals,
    'Other_Industry_Input': Other_Industry,
    'Residential_Input': Residential_Heat,
    'District_Input': District_Heat,          
    'Passenger_Input': Passenger,
    'Public_Input': Public,
    'Air_Input': Air,
    'Train_Input': Train,
    'Trucks_Input': Trucks
}

# Take the ones with True as boolean value
active_sheets = [s for s, flag in sheet_flags.items() if flag]
active_sheets

['Other_Industry_Input']

In [18]:
from shapely.ops import nearest_points

def assign_nearest_centroid(gdf, centroids):
    # Ensure everything is in same CRS
    gdf = gdf.to_crs("EPSG:3857")
    centroids = centroids.to_crs("EPSG:3857")

    # Build a spatial index for fast lookup
    centroid_sindex = centroids.sindex

    assigned_points = []

    for geom in gdf.geometry:
        possible_matches_index = list(centroid_sindex.nearest(geom.bounds, 1))
        nearest = centroids.iloc[possible_matches_index[0]].geometry
        assigned_points.append(nearest)

    gdf.loc[:, 'geometry'] = assigned_points
    return gdf.to_crs("EPSG:4326")

In [19]:
# Function to generate Voronoi polygons 
def generate_voronoi_polygons(points):
    vor = Voronoi(points.apply(lambda p: [p.x, p.y]).tolist())

    # Build the polygons
    voronoi_polygons = []
    for i, region_index in enumerate(vor.point_region):
        region = vor.regions[region_index]
        if not region or -1 in region:
            continue
        polygon = Polygon([vor.vertices[i] for i in region])
        voronoi_polygons.append(polygon)

    # Create the GeoDataframe 
    voronoi_df = gpd.GeoDataFrame(geometry=voronoi_polygons, crs="EPSG:4326")
    return voronoi_df

In [ ]:
# Read the sectors and create dataframe with precise geometries for further mapping demand
def read_sector(full_path, sheet, plz_gdf, nuts3_gdf):
    df = pd.read_excel(full_path, sheet_name=sheet)

    if {'Latitude', 'Longitude', 'Peak Load [MWh/h]'}.issubset(df.columns):
        return latlon_case(df)


    elif 'Standort-PLZ' in df.columns:
        # Merge the PLZ code with geodataframe creted earlier 
        df['Standort-PLZ'] = clean_plz(df['Standort-PLZ'])
        df = df.merge(plz_gdf[['Postleitzahl / Post code', 'geometry']], left_on='Standort-PLZ', right_on='Postleitzahl / Post code', how='left')

        missing = df[df['geometry'].isna()]
        if not missing.empty:
            print("These PLZs had no geometry match:")
            print(missing['Standort-PLZ'].unique())

        df = df.dropna(subset=['geometry'])

        # Create the geodataframe with correct geometry 
        gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')
        gdf['geometry'] = gdf['geometry'].apply(lambda geom: geom.centroid if geom.geom_type != 'Point' else geom)
        return gdf


    elif 'NUTS3' in df.columns:
        df = df.copy()
        df['NUTS3'] = df['NUTS3'].astype(str).str.strip()
        nuts3_gdf = nuts3_gdf.copy()
        nuts3_gdf['NUTS_ID'] = nuts3_gdf['NUTS_ID'].astype(str).str.strip()

        df = df.merge(nuts3_gdf, left_on='NUTS3', right_on='NUTS_ID', how='left')

        if df['geometry'].isna().any():
            missing_nuts = df[df['geometry'].isna()]['NUTS3'].unique()
            print(f"Warning: {len(missing_nuts)} NUTS3 regions had no geometry match")
            df = df.dropna(subset=['geometry'])

        if df.empty:
            return gpd.GeoDataFrame()

        gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326').to_crs('EPSG:3857')
        
        centroids = gdf.copy()
        centroids.geometry = centroids.geometry.centroid

        voronoi = generate_voronoi_polygons(centroids)
        if voronoi.empty:
            return gpd.GeoDataFrame()
        

        germany = nuts3_gdf.to_crs('EPSG:3857').unary_union
        clipped = gpd.GeoDataFrame(
            geometry=[germany.intersection(poly) for poly in voronoi.geometry],
            crs='EPSG:3857'
        ).dropna()

        gdf.geometry = clipped.representative_point()
        
        # Convert back to WGS84 (4326) for consistency
        return gdf.to_crs('EPSG:4326')

    else:
        print(f"Sheet '{sheet}' has no recognizable location format.")
        return gpd.GeoDataFrame()

In [21]:
for sheet in active_sheets:
    try:
        gdf = read_sector(full_input_path, sheet, plz_gdf, german_nuts3_gdf)

        if gdf.empty:
            print(f"Skipping '{sheet}': no valid geometries.")
            continue

        # Ensure CRS is correct
        if gdf.crs != 'EPSG:4326':
            gdf = gdf.to_crs('EPSG:4326')

        if not all(gdf.geometry.geom_type == 'Point'):
            gdf['geometry'] = gdf['geometry'].centroid
            gdf = gdf.set_geometry('geometry')

        # Extract latitude and longitude
        gdf['Latitude'] = gdf.geometry.y
        gdf['Longitude'] = gdf.geometry.x

        # Find closest node
        gdf['Closest_node'], gdf['Distance_km'] = find_closest_location(gdf, nodes_gdf)

        # Aggregate the demand
        nodes_gdf = AggregatedDemand(nodes_gdf, gdf)

    # Show possible errors
    except ValueError as e:
        print(f"Could not read sheet '{sheet}': {e}")
    except Exception as e:
        print(f"Error while processing '{sheet}': {e}")


Error while processing 'Other_Industry_Input': 'Series' object has no attribute 'x'


In [ ]:
# Nodes with demand 
non_zero_demand_nodes = nodes_gdf[nodes_gdf['Demand'] > 0]

# Take a random node as example 
node = non_zero_demand_nodes.iloc[6]
node_id = node['ID']
demand = node['Demand']

# Find all points in the data that have this node as their closest node
closest_points = gdf[gdf['Closest_node'] == node_id]

if not closest_points.empty:
    # Sum all peak loads of these closest points
    sum_peak_loads = closest_points['Peak Load [MWh/h]'].sum()

    # Compare the sum with the demand value in nodes_gdf
    if abs(sum_peak_loads - demand) < 1e-3:
        print(f"Node {node_id} has a correct demand value of {demand}.")
    else:
        print(f"The demand does not match for node {node_id} ")
else:
    print(f"No closest points found for Node {node_id}.")


IndexError: single positional indexer is out-of-bounds

In [ ]:
nodes_gdf

In [ ]:
# Check non-zero demand nodes
non_zero_demand_nodes = nodes_gdf[nodes_gdf['Demand'] > 0]
print(f"Number of nodes: {len(nodes_gdf)}")
print(f"Number of nodes with non-zero demand: {len(non_zero_demand_nodes)}")
print("Nodes with non-zero demand:")
print(non_zero_demand_nodes[['ID', 'Demand', 'Latitude', 'Longitude']])

# Summarize demand distribution
demand_summary = nodes_gdf['Demand'].describe()
print("\nDemand Distribution Summary:")
print(demand_summary)

In [ ]:
if Visualisation:
    import plotly.graph_objects as go
    import numpy as np

    df_plot = pd.DataFrame({
        'ID': nodes_gdf['ID'],
        'Longitude': nodes_gdf['Longitude'],
        'Latitude': nodes_gdf['Latitude'],
        'Demand': nodes_gdf['Demand']
    })

    # need to scale the demand for the points size on map 
    df_plot['ScaledDemand'] = df_plot['Demand'].apply(lambda d: np.log1p(d)) + 0.1
    df_plot['Used'] = df_plot['Demand'] > 0


    # Plot the map to visualize
    fig = go.Figure()

    for _, row in df_plot.iterrows():
        fig.add_trace(go.Scattergeo(
            lon=[row['Longitude']],
            lat=[row['Latitude']],
            mode='markers',
            marker=dict(
                size=max(row['ScaledDemand']*5, 2),
                color='red' if row['Used'] else 'blue',
                line=dict(width=0.5, color='black'),
                opacity=0.7
            ),
            hovertext=f"ID: {row['ID']}<br>Demand: {row['Demand']:.2f}",
            showlegend=False
        ))

    fig.update_layout(
        geo=dict(
            scope='europe',
            projection_type='natural earth',
            center=dict(lat=51.2, lon=10.4),
            lataxis=dict(range=[47, 55]),
            lonaxis=dict(range=[5, 16]),
            resolution=50
        ),
        margin={"r":0,"t":50,"l":0,"b":0},
        width=800,
        height=700,
    )

    fig.show()
